## 1. Imports

In [ ]:
from __future__ import annotations
import random
import copy
import time
import numpy as np
from pathlib import Path

from deap import base, creator, tools

# === DRY IMPORTS FROM src/notebooks/ ===
from src.notebooks.core import load_data, create_random_individual
from src.notebooks.core import course_aware_crossover, smart_mutation
from src.notebooks.core import create_evaluator, get_constraint_breakdown
from src.notebooks.core import EvolutionConfig, setup_deap, get_best_individual, EvolutionStats
from src.notebooks.viz import plot_convergence, plot_constraint_breakdown, print_summary
from src.notebooks.strategies import local_search_individual

print("✅ All imports successful!")

 All imports successful!


## 2. Mode B Configuration

In [15]:
# ============================================================================
# MODE B CONFIGURATION
# ============================================================================
from datetime import datetime

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# GA Parameters
POP_SIZE = 50
NGEN = 100
CXPB = 0.9
MUTPB = 0.2
FITNESS_WEIGHTS = (-1.0, -0.01)

# MODE B SPECIFIC: Local search parameters
LOCAL_SEARCH_PROB = 0.2        # Probability of LS per individual per gen
LOCAL_SEARCH_ITERATIONS = 10   # Iterations when LS is applied

# Paths - Organized by mode with timestamp
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_DIR = Path("../data")
OUTPUT_DIR = Path(f"../output/mode_b_memetic/{TIMESTAMP}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Mode B Config: pop={POP_SIZE}, ngen={NGEN}, LS_prob={LOCAL_SEARCH_PROB}")
print(f"📁 Output: {OUTPUT_DIR}")

✅ Mode B Config: pop=50, ngen=100, LS_prob=0.2
📁 Output: ../output/mode_b_memetic/20260120_213557


## 3. Load Data

In [16]:
data = load_data(
    data_dir=DATA_DIR,
    opening_time="10:00",
    closing_time="17:00",
    closed_days=["Saturday"],
)

print(f" {data.summary()}")

[!warn] groups enrolled but courses missing

CE604: BCE5A, BCE5B, BCE5C, BCE5D, BCE5E, BCE5F (ltp null)

ENCE 256: BCE4A, BCE4B, BCE4C, BCE4D, BCE4E, BCE4F (ltp null)

ENIE 254: BIE4A, BIE4B (ltp null)

ME706: BME7A, BME7B (ltp null)

16 course enrollments skipped

 Courses: 668, Instructors: 181, Rooms: 67, Groups: 74, Pairs: 527, Quanta: 42


## 4. Test Components

In [17]:
# Test individual creation and evaluation
evaluate = create_evaluator(data)
test_ind = create_random_individual(data)
print(f" Individual: {len(test_ind)} genes")
print(f" Initial fitness: hard={evaluate(test_ind)[0]}")

# Test local search
improved_ind, improvement = local_search_individual(
    test_ind, data, evaluate, max_iterations=5
)
print(f" After LS: hard={evaluate(improved_ind)[0]} (improvement={improvement})")

 Individual: 713 genes
 Initial fitness: hard=1248
 After LS: hard=1232 (improvement=16)


## 5. Memetic NSGA-II Evolution (Mode B Specific)

In [18]:
def run_memetic_nsga2():
    """Run NSGA-II with memetic local search."""
    print(f" Memetic NSGA-II: pop={POP_SIZE}, ngen={NGEN}, LS_prob={LOCAL_SEARCH_PROB}")
    start = time.time()
    
    # Setup DEAP
    setup_deap(FITNESS_WEIGHTS)
    
    toolbox = base.Toolbox()
    toolbox.register("individual", lambda: creator.Individual(create_random_individual(data)))
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", evaluate)
    toolbox.register("mate", course_aware_crossover)
    toolbox.register("mutate", lambda ind: smart_mutation(ind, data))
    toolbox.register("select", tools.selNSGA2)
    
    # Initialize population
    pop = toolbox.population(n=POP_SIZE)
    for ind in pop:
        ind.fitness.values = toolbox.evaluate(ind)
    
    stats = EvolutionStats()
    
    for gen in range(NGEN):
        # Standard NSGA-II selection + variation
        offspring = [copy.deepcopy(ind) for ind in toolbox.select(pop, len(pop))]
        
        # Crossover
        for i in range(0, len(offspring)-1, 2):
            if random.random() < CXPB:
                toolbox.mate(offspring[i], offspring[i+1])
                del offspring[i].fitness.values
                del offspring[i+1].fitness.values
        
        # Mutation
        for ind in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(ind)
                del ind.fitness.values
        
        # === MODE B SPECIFIC: Local Search ===
        for ind in offspring:
            if random.random() < LOCAL_SEARCH_PROB:
                genes = list(ind)
                improved_genes, _ = local_search_individual(
                    genes, data, evaluate, LOCAL_SEARCH_ITERATIONS
                )
                ind[:] = improved_genes
                del ind.fitness.values
        
        # Evaluate
        for ind in offspring:
            if not ind.fitness.valid:
                ind.fitness.values = toolbox.evaluate(ind)
        
        # Survivor selection
        pop = toolbox.select(pop + offspring, POP_SIZE)
        
        # Stats
        hard_vals = [ind.fitness.values[0] for ind in pop]
        soft_vals = [ind.fitness.values[1] for ind in pop]
        stats.generations.append(gen)
        stats.min_hard.append(float(min(hard_vals)))
        stats.avg_hard.append(float(np.mean(hard_vals)))
        stats.max_hard.append(float(max(hard_vals)))
        stats.feasible_count.append(sum(1 for h in hard_vals if h == 0))
        stats.min_soft.append(float(min(soft_vals)))
        stats.avg_soft.append(float(np.mean(soft_vals)))
        
        if gen % 20 == 0 or gen == NGEN-1:
            print(f"  Gen {gen:3d}: min_hard={stats.min_hard[-1]:3.0f}, min_soft={stats.min_soft[-1]:5.0f}, feasible={stats.feasible_count[-1]}/{POP_SIZE}")
    
    stats.elapsed_time = time.time() - start
    print(f" Done in {stats.elapsed_time:.1f}s")
    return pop, stats

# RUN
final_pop, stats = run_memetic_nsga2()

 Memetic NSGA-II: pop=50, ngen=100, LS_prob=0.2
  Gen   0: min_hard=1246, min_soft=  771, feasible=0/50
  Gen  20: min_hard=1118, min_soft=  650, feasible=0/50
  Gen  40: min_hard=1010, min_soft=  533, feasible=0/50
  Gen  60: min_hard=937, min_soft=  481, feasible=0/50
  Gen  80: min_hard=851, min_soft=  400, feasible=0/50
  Gen  99: min_hard=791, min_soft=  362, feasible=0/50
 Done in 353.1s


## 6. Results & Visualization

In [19]:
best = get_best_individual(final_pop)
breakdown = get_constraint_breakdown(best, data)

print_summary(final_pop, stats, breakdown)

plot_convergence(stats, OUTPUT_DIR / "mode_b_convergence.png", title_prefix="Mode B: ")
plot_constraint_breakdown(breakdown, OUTPUT_DIR / "mode_b_breakdown.png", title="Mode B: Constraint Violations")


 RESULTS SUMMARY
Best Solution: hard=791.0, soft=512.0
Final Generation: min_hard=791, min_soft=362, avg_hard=841.4
Feasible Solutions: 0
Elapsed Time: 353.1s

Hard Constraint Violations:
  student_group_exclusivity: 461
  instructor_exclusivity: 87
  instructor_qualifications: 0
  room_exclusivity: 243
  room_suitability: 0

Soft Constraint Penalties:
  student_schedule_compactness: 230
  instructor_schedule_compactness: 90
  student_lunch_break: 192
 Saved: ../output/mode_b_memetic/20260120_213557/mode_b_convergence.png
 Saved: ../output/mode_b_memetic/20260120_213557/mode_b_breakdown.png


<Figure size 1000x500 with 1 Axes>